# ACLE Phase 0 — Operator-Expansion Ceiling Test (hard gate)

Thin launcher. All logic lives in `src/` (protocol + decision rule: `src/phase0.py`, settings: `configs/phase0_ceiling_test.yaml`).

**Before starting:** `Runtime > Change runtime type > T4 GPU`.

**Every session (incl. after a disconnect):** run cells 1 and 2, then whichever step you are on. Every step resumes where it stopped, losing at most one training run.

| Step | Cell | When |
|---|---|---|
| Setup | 1, 2 | every session |
| Timing benchmark (~2 min) | 3 | first session only; paste the printed line to Claude |
| Evolution runs (10 runs) | 4 | until it prints *All evolution runs complete* |
| Final held-out evaluation | 5 | after step 4 |
| Analysis + PASS/FAIL | 6 | after step 5; paste the marked block to Claude |

**Free tier tips:** keep this tab open (idle tabs get disconnected); when you are done for the day use `Runtime > Disconnect and delete runtime`.

In [ ]:
# === 1. Setup (every session) ===
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BRANCH = 'claude/stoic-knuth-jenuuo'
REPO_DIR = '/content/GP-Loss'
REPO_URL = 'https://github.com/danielronak/GP-Loss.git'
# Private repo? Add a GH_TOKEN secret (key icon in the left sidebar), then uncomment:
# from google.colab import userdata
# REPO_URL = f"https://{userdata.get('GH_TOKEN')}@github.com/danielronak/GP-Loss.git"

if not os.path.exists(REPO_DIR):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q origin {BRANCH}
!git -C {REPO_DIR} log -1 --format='code version: %h  %s'
!pip install -q deap dill pyyaml

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE -> Runtime > Change runtime type > T4 GPU')

In [ ]:
# === 2. Config + data (every session) ===
from src import phase0

cfg = phase0.load_config('configs/phase0_ceiling_test.yaml')
OUTPUT_DIR = '/content/drive/MyDrive/ACLE/phase0'   # checkpoints + results (on Drive)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Checkpoints/results ->', os.path.abspath(OUTPUT_DIR))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
data = phase0.prepare_data(cfg, data_root='/content/data', device=device)
print({k: len(v) for k, v in data.items()})

In [ ]:
# === 3. Timing benchmark (first session only, ~2 min) ===
# Paste the printed line to Claude so the budget can be checked before the long run.
est = phase0.benchmark(cfg, data)

In [ ]:
# === 4. Evolution runs (resumable: just re-run after a disconnect) ===
# SESSION_BUDGET_MIN stops cleanly after N minutes (None = run until done).
SESSION_BUDGET_MIN = None
phase0.run_evolutions(cfg, data, OUTPUT_DIR, time_budget_minutes=SESSION_BUDGET_MIN)

In [ ]:
# === 5. Final held-out evaluation (resumable) ===
# Retrains each run's best tree + CrossEntropy + MSE on the full train split,
# scored on the untouched test set, 5 shared seeds each.
phase0.run_final_eval(cfg, data, OUTPUT_DIR, time_budget_minutes=SESSION_BUDGET_MIN)

In [ ]:
# === 6. Analysis + PASS/FAIL (writes phase0_result.json next to the checkpoints) ===
phase0.write_runs_csv(cfg, OUTPUT_DIR)
result = phase0.analyze_dir(cfg, OUTPUT_DIR)
# Then: Runtime > Disconnect and delete runtime